# Feature generation | Google Earth Engine

Public adaptation of `src/05_mapping/03_multi_waterbody_moving_window_features.ipynb` in the team archive. Use this to inspect how features were produced or to regenerate them with your own Earth Engine project and authorized inputs. To use the existing exports, go directly to the prediction notebooks instead. Exports are disabled by default. Details: [Feature sources and Drive inputs](https://github.com/Boonyaratt/sentinel2-water-quality/blob/main/docs/drive-features-th.md).


# Multi-Waterbody Moving-Window Feature Pipeline (Google Colab)

This notebook processes every monitored water body that has at least one `clear_water_ready=True` date from Notebook 02. By default, it selects the best date independently for each water body, builds the same Sentinel-2 water mask used during tabular extraction, and creates 100 m moving-window feature rasters.

The notebook exports model-input feature stacks, not final prediction maps. Prediction is deliberately gated until the exported raster features pass schema, range, and station-point checks.

Pipeline:

1. Load the corrected `static_intersection_v2` coverage table and context/station table.
2. Select the best ready date per water body.
3. Build a station-supported AOI for every selected water body.
4. Create daily Sentinel-2 mosaics, clear/hybrid-water masks, moving-window medians, QA layers, and selected environmental context layers.
5. Audit the band schema and sample the stacks at station locations.
6. Export one multiband GeoTIFF per `waterbody x date` to Google Drive.

The AOIs are station-supported mapping extents, not complete hydrological boundaries.

In [ ]:
# Colab setup.
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import subprocess
import sys

required = {"ee": "earthengine-api", "geemap": "geemap", "rasterio": "rasterio"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from pathlib import Path
import re
import time
import unicodedata

import ee
import geemap
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## 1. Configuration and Inputs

Use `TOP_N_DATES_PER_WATERBODY = 1` for one consistent date per water body. Increase it only for sensitivity analysis. `MAX_WATERBODIES = None` processes every eligible water body; set an integer for a short test run.

In [ ]:
MYDRIVE = Path("/content/drive/MyDrive")
PROJECT_DIR = MYDRIVE / "wqr_feature_rebuild"

# Optional complete paths. Keep None for automatic lookup in /content and MyDrive.
COVERAGE_PATH_OVERRIDE = None
CONTEXT_PATH_OVERRIDE = None
MODEL_DATA_PATH_OVERRIDE = None  # Optional; enables training-range audit.
POLYGON_PATH_OVERRIDE = None  # Combined GeoJSON from Notebook 02b.

COVERAGE_PATTERN = "clear_water_coverage_by_date_waterbody*.csv"
CONTEXT_FILENAME = "rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv"
MODEL_DATA_FILENAME = "rswq_model_input_relaxed_s2_context_2562_2568.csv"
POLYGON_FILENAME = "waterbody_polygons.geojson"

EE_PROJECT = "YOUR_EARTH_ENGINE_PROJECT_ID"
if EE_PROJECT == "YOUR_EARTH_ENGINE_PROJECT_ID":
    raise ValueError("Set EE_PROJECT to a project you can use in Earth Engine.")
S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
GSW_IMAGE = "JRC/GSW1_4/GlobalSurfaceWater"
GSMAP_COLLECTION = "JAXA/GPM_L3/GSMaP/v8/operational"
ERA5_COLLECTION = "ECMWF/ERA5_LAND/HOURLY"
WORLDCOVER_COLLECTION = "ESA/WorldCover/v200"

TOP_N_DATES_PER_WATERBODY = 1
MAX_WATERBODIES = None
DATE_OVERRIDE_BY_WATERBODY = {"ทะเลน้อย": "2024-07-19"}
EXPORT_MAP_IDS = ["WB01", "WB02"]  # Full reviewed lake polygons; station-buffer exports are prohibited.
INCLUDE_WATERBODIES = None  # Example: ["แม่น้ำชุมพร", "ทะเลหลวง"]

MOVING_WINDOW_RADIUS_M = 100
EXPORT_SCALE_M = 20
MIN_WINDOW_WATER_PIXELS = 3
MAX_SCENE_CLOUD_PCT = 80
JRC_OCCURRENCE_THRESHOLD = 70

DRIVE_EXPORT_FOLDER = "Water_Quality_RS_Moving_Window_Polygon_rebuild"
RGB_DRIVE_EXPORT_FOLDER = "Water_Quality_RS_Sentinel2_RGB_Polygon_rebuild"
OUTPUT_DIR = PROJECT_DIR / "Maps" / "moving_window"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_EXPORT_TASKS = False
WAIT_FOR_EXPORTS = False
MAX_WAIT_MINUTES = 180
POLL_SECONDS = 30


def resolve_file(override, exact_name=None, pattern=None, required=True):
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    roots = [Path("/content"), PROJECT_DIR, MYDRIVE / "Water Quality Research"]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        if exact_name:
            matches.extend(root.rglob(exact_name))
        elif pattern:
            matches.extend(root.rglob(pattern))

    matches = sorted(set(matches), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        if not required:
            return None
        label = exact_name or pattern
        print(f"Could not find {label}. Upload it now.")
        uploaded = files.upload()
        uploaded_paths = [Path("/content") / name for name in uploaded]
        if exact_name:
            matches = [path for path in uploaded_paths if path.name == exact_name]
        else:
            import fnmatch
            matches = [path for path in uploaded_paths if fnmatch.fnmatch(path.name, pattern)]
        if not matches:
            raise FileNotFoundError(
                f"The uploaded files did not match {label}. Set the corresponding *_PATH_OVERRIDE."
            )
    if len(matches) > 1:
        print("Multiple files found; using the newest:")
        for path in matches[:10]:
            print(" -", path)
    return matches[0]


COVERAGE_PATH = resolve_file(COVERAGE_PATH_OVERRIDE, pattern=COVERAGE_PATTERN)
CONTEXT_PATH = resolve_file(CONTEXT_PATH_OVERRIDE, exact_name=CONTEXT_FILENAME)
MODEL_DATA_PATH = resolve_file(MODEL_DATA_PATH_OVERRIDE, exact_name=MODEL_DATA_FILENAME, required=False)
POLYGON_PATH = resolve_file(POLYGON_PATH_OVERRIDE, exact_name=POLYGON_FILENAME, required=True)

print("Coverage:", COVERAGE_PATH)
print("Context/stations:", CONTEXT_PATH)
print("Optional model data:", MODEL_DATA_PATH)
print("Reviewed polygons:", POLYGON_PATH)
print("Output directory:", OUTPUT_DIR)


## 2. Earth Engine Authentication

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized:", EE_PROJECT)

## 3. Build the Multi-Waterbody Mapping Manifest

Selection is performed independently within each water body. Ready candidates are ordered by hybrid clear-water coverage, clear coverage, retained water pixels, and date. The first row per water body becomes its locked visualization date.

In [ ]:
coverage = pd.read_csv(COVERAGE_PATH, low_memory=False)
context = pd.read_csv(
    CONTEXT_PATH,
    usecols=[
        "sample_id", "station_canonical", "coordinate_waterbody",
        "latitude", "longitude", "field_date", "year_be",
    ],
    low_memory=False,
)

required_coverage = {
    "sat_date", "coordinate_waterbody", "screening_metric_version",
    "clear_static_pixels", "clear_static_water_pixels",
    "clear_static_coverage_pct", "clear_static_water_coverage_pct",
    "eligible_static_water", "clear_water_ready",
}
missing = required_coverage.difference(coverage.columns)
if missing:
    raise KeyError(f"Coverage table is missing: {sorted(missing)}")

versions = set(coverage["screening_metric_version"].dropna().astype(str))
if versions != {"static_intersection_v2"}:
    raise ValueError(f"Expected only static_intersection_v2, found: {versions}")

coverage["sat_date"] = pd.to_datetime(coverage["sat_date"], errors="coerce").dt.normalize()
for column in ["latitude", "longitude"]:
    context[column] = pd.to_numeric(context[column], errors="coerce")

ready = coverage.loc[
    coverage["eligible_static_water"].eq(True)
    & coverage["clear_water_ready"].eq(True)
    & coverage["sat_date"].notna()
].copy()

if INCLUDE_WATERBODIES is not None:
    ready = ready.loc[ready["coordinate_waterbody"].isin(INCLUDE_WATERBODIES)].copy()

ready = ready.sort_values(
    [
        "coordinate_waterbody", "clear_static_water_coverage_pct",
        "clear_static_coverage_pct", "clear_static_water_pixels", "sat_date",
    ],
    ascending=[True, False, False, False, True],
)
ready["candidate_rank_within_waterbody"] = ready.groupby("coordinate_waterbody").cumcount() + 1
manifest = ready.loc[
    ready["candidate_rank_within_waterbody"].le(TOP_N_DATES_PER_WATERBODY)
].copy()

if MAX_WATERBODIES is not None:
    selected_names = manifest["coordinate_waterbody"].drop_duplicates().head(MAX_WATERBODIES)
    manifest = manifest.loc[manifest["coordinate_waterbody"].isin(selected_names)].copy()

# Replace the automatically ranked date only when an explicit reviewed override is supplied.
for waterbody, date_text in DATE_OVERRIDE_BY_WATERBODY.items():
    override_date = pd.Timestamp(date_text).normalize()
    candidate = ready.loc[
        ready["coordinate_waterbody"].eq(waterbody)
        & ready["sat_date"].eq(override_date)
    ].copy()
    if len(candidate) != 1:
        raise ValueError(
            f"Expected one ready date override for {waterbody} on {date_text}, found {len(candidate)}"
        )
    manifest = pd.concat([
        manifest.loc[~manifest["coordinate_waterbody"].eq(waterbody)],
        candidate,
    ], ignore_index=True)

manifest = manifest.sort_values(["coordinate_waterbody", "candidate_rank_within_waterbody"]).reset_index(drop=True)
manifest.insert(0, "map_id", [f"WB{i:02d}" for i in range(1, len(manifest) + 1)])

station_inventory = (
    context.dropna(subset=["coordinate_waterbody", "station_canonical", "latitude", "longitude"])
    .drop_duplicates(["coordinate_waterbody", "station_canonical", "latitude", "longitude"])
)
station_counts = station_inventory.groupby("coordinate_waterbody")["station_canonical"].nunique()
manifest["n_station_points"] = manifest["coordinate_waterbody"].map(station_counts).fillna(0).astype(int)

if manifest.empty:
    raise ValueError("No clear_water_ready candidates remain after configuration filters.")

if manifest["n_station_points"].eq(0).any():
    raise ValueError("At least one selected water body has no valid station coordinates.")

manifest_path = OUTPUT_DIR / "mapping_manifest.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

display(manifest[[
    "map_id", "coordinate_waterbody", "sat_date", "candidate_rank_within_waterbody",
    "n_station_points", "clear_static_coverage_pct",
    "clear_static_water_coverage_pct", "clear_static_water_pixels",
]].round(2))
print("Selected water bodies:", manifest["coordinate_waterbody"].nunique())
print("Selected waterbody-date jobs:", len(manifest))
print("Manifest:", manifest_path)

## Reviewed Polygon AOIs

In [ ]:
# Validate and load the reviewed polygon AOIs from Notebook 02b.
import json

polygon_geojson = json.loads(Path(POLYGON_PATH).read_text(encoding="utf-8"))
polygon_features = polygon_geojson.get("features", [])
polygon_rows = []
for feature in polygon_features:
    properties = feature.get("properties", {})
    geometry = feature.get("geometry") or {}
    polygon_rows.append({
        "map_id": str(properties.get("map_id")),
        "coordinate_waterbody": properties.get("coordinate_waterbody"),
        "geometry_type": geometry.get("type"),
        "has_coordinates": bool(geometry.get("coordinates")),
    })
polygon_audit = pd.DataFrame(polygon_rows)

if polygon_audit["map_id"].duplicated().any():
    raise ValueError("Polygon GeoJSON contains duplicate map_id features.")
missing_polygon_ids = sorted(set(manifest["map_id"]) - set(polygon_audit["map_id"]))
unexpected_polygon_ids = sorted(set(polygon_audit["map_id"]) - set(manifest["map_id"]))
if missing_polygon_ids or unexpected_polygon_ids:
    raise ValueError(
        f"Polygon/manifest mismatch: missing={missing_polygon_ids}, unexpected={unexpected_polygon_ids}"
    )
if not polygon_audit["geometry_type"].isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Every AOI must be Polygon or MultiPolygon.")
if not polygon_audit["has_coordinates"].all():
    raise ValueError("At least one polygon has empty coordinates.")

WATERBODY_POLYGONS = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry(feature["geometry"]),
        feature.get("properties", {}),
    )
    for feature in polygon_features
])
display(polygon_audit.sort_values("map_id"))
print("Polygon AOIs ready:", len(polygon_audit))

## 4. Sentinel-2 Preprocessing and Hybrid Water Mask

This reproduces the scaling, SCL exclusions, spectral indices, bridge/built-up rejection, and connected hybrid-water rule used in the tabular extraction notebooks.

In [ ]:
S2_BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]
BASE_FEATURES = S2_BANDS + ["NDWI", "MNDWI", "NDTI", "NDCI", "NDVI", "AWEIsh"]


def safe_ratio(numerator, denominator, name):
    return numerator.divide(denominator).updateMask(denominator.neq(0)).rename(name)


def normalized_difference(a, b, name):
    return a.subtract(b).divide(a.add(b)).updateMask(a.add(b).neq(0)).rename(name)


def prepare_s2(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8))
        .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    )
    sr = image.select(S2_BANDS).multiply(0.0001).rename(S2_BANDS)
    blue = sr.select("B2")
    green = sr.select("B3")
    red = sr.select("B4")
    red_edge1 = sr.select("B5")
    nir = sr.select("B8")
    swir1 = sr.select("B11")
    swir2 = sr.select("B12")

    ndvi = normalized_difference(nir, red, "NDVI")
    ndwi = normalized_difference(green, nir, "NDWI")
    mndwi = normalized_difference(green, swir1, "MNDWI")
    ndti = normalized_difference(red, green, "NDTI")
    ndci = normalized_difference(red_edge1, red, "NDCI")
    ndbi = normalized_difference(swir1, nir, "NDBI")
    aweish = (
        blue.add(green.multiply(2.5))
        .subtract(nir.add(swir1).multiply(1.5))
        .subtract(swir2.multiply(0.25))
        .rename("AWEIsh")
    )
    brightness = blue.add(green).add(red).divide(3)

    built_or_bridge = (
        brightness.gt(0.12).And(mndwi.lt(0.12)).And(ndwi.lt(0.12)).And(aweish.lt(0.08))
    ).Or(
        ndbi.gt(0).And(mndwi.lt(0.15))
    ).Or(
        nir.gt(0.20).And(swir1.gt(0.18)).And(mndwi.lt(-0.10)).And(scl.neq(6))
    )

    jrc_water = ee.Image(GSW_IMAGE).select("occurrence").gte(JRC_OCCURRENCE_THRESHOLD)
    jrc_current = (
        jrc_water.And(scl.eq(6).Or(ndwi.gt(0.03)).Or(mndwi.gt(0.03)).Or(aweish.gt(0.03)))
        .And(nir.lt(0.25)).And(swir1.lt(0.18)).And(ndvi.lt(0.30)).And(built_or_bridge.Not())
    )
    strong = mndwi.gt(0.20).And(aweish.gt(0.05)).And(nir.lt(0.12)).And(swir1.lt(0.10)).And(ndvi.lt(0.15))
    turbid = mndwi.gt(-0.20).And(ndwi.gt(-0.30)).And(aweish.gt(-0.10)).And(nir.lt(0.20)).And(swir1.lt(0.16)).And(ndvi.lt(0.30))
    scl_current = (
        scl.eq(6).And(nir.lt(0.16)).And(swir1.lt(0.12)).And(ndvi.lt(0.22))
        .And(mndwi.gt(0).Or(ndwi.gt(-0.05)).Or(aweish.gt(-0.02))).And(built_or_bridge.Not())
    )
    index_core = strong.Or(turbid.And(mndwi.gt(0.05).Or(aweish.gt(0.02)))).And(built_or_bridge.Not())
    water_core = scl_current.Or(jrc_current).Or(index_core).And(built_or_bridge.Not())
    water_candidate = turbid.And(built_or_bridge.Not())
    core_neighborhood = water_core.focal_max(radius=2, units="pixels")
    water_mask = water_core.Or(water_candidate.And(core_neighborhood)).And(built_or_bridge.Not()).updateMask(clear)

    return (
        sr.addBands([ndwi, mndwi, ndti, ndci, ndvi, aweish])
        .addBands(ee.Image(1).rename("valid_mask").updateMask(clear))
        .addBands(water_mask.rename("water_mask"))
        .copyProperties(image, ["system:index", "system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])
    )

## 5. Moving-Window, QA, and Environmental Context Features

In [ ]:
# Every neighborhood uses an explicit pixel kernel on an explicit projection.
# This prevents GEE from converting a meter kernel on an unintended native scale.
MOVING_WINDOW_RADIUS_PIXELS = int(np.ceil(MOVING_WINDOW_RADIUS_M / EXPORT_SCALE_M))
WINDOW_KERNEL = ee.Kernel.circle(
    radius=MOVING_WINDOW_RADIUS_PIXELS, units="pixels", normalize=False
)
JRC_CONTEXT_SCALE_M = 30
JRC_500_RADIUS_PIXELS = int(np.ceil(500 / JRC_CONTEXT_SCALE_M))
JRC_500_KERNEL = ee.Kernel.circle(
    radius=JRC_500_RADIUS_PIXELS, units="pixels", normalize=False
)
COARSE_CONTEXT_SCALE_M = 11_132
WORLDCOVER = ee.ImageCollection(WORLDCOVER_COLLECTION).first().select("Map")
# Aggregate 10 m WorldCover indicators to 50 m before large focal windows.
# A direct 5 km radius at 10 m creates a 1001-pixel kernel; GEE allows at most 512.
WORLDCOVER_FRACTION_SCALE_M = 50
JRC = ee.Image(GSW_IMAGE)
GSMAP = ee.ImageCollection(GSMAP_COLLECTION).select("hourlyPrecipRateGC")
ERA5 = ee.ImageCollection(ERA5_COLLECTION)


def focal_mean_coarse(image, radius_m, output_name):
    radius_pixels = int(np.ceil(radius_m / COARSE_CONTEXT_SCALE_M))
    if 2 * radius_pixels + 1 > 512:
        raise ValueError(f"Coarse kernel too large for {output_name}: {2 * radius_pixels + 1} pixels")
    kernel = ee.Kernel.circle(radius=radius_pixels, units="pixels", normalize=False)
    projected = image.reproject(crs="EPSG:4326", scale=COARSE_CONTEXT_SCALE_M)
    return projected.reduceNeighborhood(
        ee.Reducer.mean(), kernel, skipMasked=False
    ).rename(output_name)


def make_static_context():
    jrc_projection = JRC.select("occurrence").projection()
    jrc_occurrence_30m = JRC.select("occurrence").reproject(
        crs=jrc_projection, scale=JRC_CONTEXT_SCALE_M
    )
    jrc_recurrence_30m = JRC.select("recurrence").reproject(
        crs=jrc_projection, scale=JRC_CONTEXT_SCALE_M
    )
    bands = [
        jrc_occurrence_30m.reduceNeighborhood(
            ee.Reducer.mean(), JRC_500_KERNEL, skipMasked=False
        ).rename("jrc_occurrence_mean_500m"),
        jrc_recurrence_30m.reduceNeighborhood(
            ee.Reducer.mean(), JRC_500_KERNEL, skipMasked=False
        ).rename("jrc_recurrence_mean_500m"),
    ]
    selected_wc = {
        ("tree", 10, 500), ("tree", 10, 1000),
        ("shrub", 20, 500), ("shrub", 20, 1000), ("shrub", 20, 5000),
        ("grass", 30, 1000), ("crop", 40, 1000),
        ("bare", 60, 500), ("bare", 60, 1000),
        ("water", 80, 500), ("water", 80, 1000), ("water", 80, 5000),
        ("wetland", 90, 5000),
    }
    for name, class_value, radius in sorted(selected_wc):
        # Mean aggregation preserves class fraction while keeping the 5 km kernel below GEE's limit.
        class_fraction_50m = (
            WORLDCOVER.eq(class_value).unmask(0).toFloat()
            .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
            .reproject(crs=WORLDCOVER.projection(), scale=WORLDCOVER_FRACTION_SCALE_M)
        )
        # Use an explicit pixel kernel so GEE cannot reinterpret 5 km on the native 10 m grid.
        radius_pixels = int(np.ceil(radius / WORLDCOVER_FRACTION_SCALE_M))
        fraction_kernel = ee.Kernel.circle(
            radius=radius_pixels, units="pixels", normalize=False
        )
        bands.append(
            class_fraction_50m.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=fraction_kernel,
                skipMasked=False,
            ).rename(f"wc_{name}_frac_{radius}m")
        )
    return ee.Image.cat(bands)



MAX_KERNEL_DIAMETERS = {
    "moving_window": 2 * MOVING_WINDOW_RADIUS_PIXELS + 1,
    "jrc_500m": 2 * JRC_500_RADIUS_PIXELS + 1,
    "worldcover_5000m": 2 * int(np.ceil(5000 / WORLDCOVER_FRACTION_SCALE_M)) + 1,
    "coarse_15000m": 2 * int(np.ceil(15000 / COARSE_CONTEXT_SCALE_M)) + 1,
}
if max(MAX_KERNEL_DIAMETERS.values()) > 512:
    raise ValueError(f"Kernel guard failed: {MAX_KERNEL_DIAMETERS}")
print("Explicit kernel diameters (pixels):", MAX_KERNEL_DIAMETERS)

STATIC_CONTEXT = make_static_context()


def prep_era5(image):
    air = image.select("temperature_2m").subtract(273.15).rename("air_temp_C")
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed_m_s")
    runoff = image.select("surface_runoff_hourly").multiply(1000).rename("runoff_mm")
    solar = image.select("surface_solar_radiation_downwards_hourly").divide(1e6).rename("solar_MJ_m2")
    return ee.Image.cat([air, wind, runoff, solar]).copyProperties(image, ["system:time_start"])


def make_temporal_context(cutoff):
    # Only complete hours before the Sentinel-2 acquisition-hour cutoff are used.
    rain24 = GSMAP.filterDate(cutoff.advance(-24, "hour"), cutoff).sum()
    rain336 = GSMAP.filterDate(cutoff.advance(-336, "hour"), cutoff)
    rain720 = GSMAP.filterDate(cutoff.advance(-720, "hour"), cutoff).sum()
    rainy336 = rain336.map(lambda img: img.gt(1.0).rename("rainy")).sum()

    era1 = ERA5.filterDate(cutoff.advance(-1, "hour"), cutoff).map(prep_era5)
    era24 = ERA5.filterDate(cutoff.advance(-24, "hour"), cutoff).map(prep_era5)
    era72 = ERA5.filterDate(cutoff.advance(-72, "hour"), cutoff).map(prep_era5)
    era168 = ERA5.filterDate(cutoff.advance(-168, "hour"), cutoff).map(prep_era5)

    coarse = ee.Image.cat([
        focal_mean_coarse(rain24, 10_000, "rain_gsmap_prior_24h_mm"),
        focal_mean_coarse(rain336.sum(), 10_000, "rain_gsmap_prior_336h_mm"),
        focal_mean_coarse(rain720, 10_000, "rain_gsmap_prior_720h_mm"),
        focal_mean_coarse(rainy336, 10_000, "rainy_hours_gsmap_prior_336h"),
        focal_mean_coarse(era1.sum().select("runoff_mm"), 15_000, "era5h_runoff_prior_1h_mm"),
        focal_mean_coarse(era24.sum().select("runoff_mm"), 15_000, "era5h_runoff_prior_24h_mm"),
        focal_mean_coarse(era72.mean().select("air_temp_C"), 15_000, "era5h_air_temp_prior_72h_C"),
        focal_mean_coarse(era168.max().select("wind_speed_m_s"), 15_000, "era5h_wind_max_prior_168h_m_s"),
        focal_mean_coarse(era168.sum().select("solar_MJ_m2"), 15_000, "era5h_solar_prior_168h_MJ_m2"),
    ])
    month = ee.Image.constant(ee.Number.parse(cutoff.format("M"))).rename("month").toFloat()
    doy = ee.Image.constant(cutoff.getRelative("day", "year").add(1)).rename("day_of_year").toFloat()
    return coarse.addBands([month, doy])


def make_feature_stack(row, geometry):
    sat_date = pd.Timestamp(row["sat_date"])
    start = ee.Date(sat_date.strftime("%Y-%m-%d"))
    end = start.advance(1, "day")
    collection = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(geometry)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    )
    if collection.size().getInfo() == 0:
        raise ValueError(f"No S2 scenes for {row['map_id']} on {sat_date.date()}")

    prepared = collection.map(prepare_s2).mosaic()
    target_projection = prepared.select("B3").projection()
    water = (
        prepared.select("water_mask").unmask(0).gt(0)
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
    )
    clear = (
        prepared.select("valid_mask").unmask(0).gt(0)
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
    )
    base_water = (
        prepared.select(BASE_FEATURES).resample("bilinear")
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
        .updateMask(water)
    )

    moving = base_water.reduceNeighborhood(
        reducer=ee.Reducer.median(), kernel=WINDOW_KERNEL, skipMasked=True
    ).rename([f"{name}_median" for name in BASE_FEATURES])

    one = ee.Image.constant(1).rename("one").reproject(
        crs=target_projection, scale=EXPORT_SCALE_M
    )
    total_count = one.reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("total_pixel_count")
    clear_count = clear.rename("clear").reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("valid_pixel_count")
    water_count = water.rename("water").reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("valid_water_pixel_count")
    water_fraction = water_count.divide(clear_count.max(1)).clamp(0, 1).rename("water_pixel_fraction")
    water_pct = water_fraction.multiply(100).rename("water_pixel_pct")
    local_cloud_pct = ee.Image(1).subtract(clear_count.divide(total_count)).clamp(0, 1).multiply(100).rename("local_cloud_pct")
    candidate_water_count = water_count.rename("candidate_water_count")
    scene_cloud_pct = ee.Image.constant(collection.aggregate_mean("CLOUDY_PIXEL_PERCENTAGE")).rename("scene_cloud_pct")

    # Custom SS features are computed from moving-window band medians, matching the pandas training code.
    b2, b3, b4 = moving.select("B2_median"), moving.select("B3_median"), moving.select("B4_median")
    b8, b11, b12 = moving.select("B8_median"), moving.select("B11_median"), moving.select("B12_median")
    custom = ee.Image.cat([
        normalized_difference(b3, b8, "NDWI1"),
        normalized_difference(b8, b11, "NDWI2"),
        normalized_difference(b8, b12, "NDWI3"),
        safe_ratio(b4, b3, "B4_B3_ratio"),
        safe_ratio(b8, b4, "B8_B4_ratio"),
        safe_ratio(b8, b2, "NIR_Blue_ratio"),
        safe_ratio(b8, b3, "NIR_Green_ratio"),
        safe_ratio(b8, b4, "NIR_Red_ratio"),
        normalized_difference(b4, b8, "Red_NIR_ND"),
    ])

    acquisition = ee.Date(collection.aggregate_min("system:time_start"))
    cutoff = ee.Date.parse("YYYY-MM-dd HH", acquisition.format("YYYY-MM-dd HH"))
    temporal_context = make_temporal_context(cutoff)

    qa = ee.Image.cat([
        water_count, clear_count, total_count, water_fraction, water_pct,
        candidate_water_count, local_cloud_pct, scene_cloud_pct,
    ])
    prediction_support = water.And(water_count.gte(MIN_WINDOW_WATER_PIXELS)).rename("prediction_support")

    stack = (
        moving.addBands(custom).addBands(qa)
        .addBands(STATIC_CONTEXT).addBands(temporal_context)
        .addBands(prediction_support)
        .updateMask(prediction_support)
        .clip(geometry)
        .toFloat()
    )
    metadata = {
        "n_source_scenes": collection.size().getInfo(),
        "source_scene_ids": collection.aggregate_array("system:index").getInfo(),
        "acquisition_cutoff_utc": cutoff.format("YYYY-MM-dd'T'HH:mm:ss").getInfo(),
    }
    return stack, prepared, metadata


## 6. Build All Ready-Area Stacks and Audit the Schema

In [ ]:
def geometry_and_points(map_id, waterbody):
    subset = station_inventory.loc[station_inventory["coordinate_waterbody"].eq(waterbody)].copy()
    matches = WATERBODY_POLYGONS.filter(ee.Filter.eq("map_id", str(map_id)))
    match_count = matches.size().getInfo()
    if match_count != 1:
        raise ValueError(f"Expected one polygon for {map_id}, found {match_count}")
    geometry = ee.Feature(matches.first()).geometry()
    features = [
        ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {
            "station_canonical": row.station_canonical,
            "coordinate_waterbody": row.coordinate_waterbody,
        })
        for row in subset.itertuples()
    ]
    return geometry, ee.FeatureCollection(features)


jobs = {}
job_rows = []
expected_schema = None

for row in manifest.to_dict("records"):
    geometry, station_points = geometry_and_points(row["map_id"], row["coordinate_waterbody"])
    stack, prepared, metadata = make_feature_stack(row, geometry)
    schema = stack.bandNames().getInfo()
    if expected_schema is None:
        expected_schema = schema
    elif schema != expected_schema:
        raise ValueError(f"Band schema mismatch for {row['map_id']}")

    jobs[row["map_id"]] = {
        "row": row, "geometry": geometry, "points": station_points,
        "stack": stack, "prepared": prepared, "metadata": metadata,
    }
    job_rows.append({
        "map_id": row["map_id"],
        "coordinate_waterbody": row["coordinate_waterbody"],
        "sat_date": pd.Timestamp(row["sat_date"]).date().isoformat(),
        "aoi_source": "reviewed_drawn_polygon",
        "aoi_area_km2": geometry.area(maxError=1).divide(1e6).getInfo(),
        "n_bands": len(schema),
        **metadata,
    })
    print(row["map_id"], row["coordinate_waterbody"], pd.Timestamp(row["sat_date"]).date(), "bands:", len(schema))

job_audit = pd.DataFrame(job_rows)
band_schema = pd.DataFrame({"band_order": range(1, len(expected_schema) + 1), "band": expected_schema})
job_audit.to_csv(OUTPUT_DIR / "moving_window_job_audit.csv", index=False, encoding="utf-8-sig")
band_schema.to_csv(OUTPUT_DIR / "moving_window_band_schema.csv", index=False, encoding="utf-8-sig")

display(job_audit)
display(band_schema)

## 7. Interactive Coverage Preview

Each layer uses its own locked date. This preview is for mask/AOI inspection; it is not a model-performance validation.

In [ ]:
center_lat = station_inventory.loc[
    station_inventory["coordinate_waterbody"].isin(manifest["coordinate_waterbody"]), "latitude"
].mean()
center_lon = station_inventory.loc[
    station_inventory["coordinate_waterbody"].isin(manifest["coordinate_waterbody"]), "longitude"
].mean()

Map = geemap.Map(center=[center_lat, center_lon], zoom=7, height="720px")
Map.add_basemap("Esri.WorldImagery")

for map_id, job in jobs.items():
    row = job["row"]
    date_label = pd.Timestamp(row["sat_date"]).date().isoformat()
    clear_rgb = (
        job["prepared"].select(["B4", "B3", "B2"])
        .updateMask(job["prepared"].select("valid_mask"))
        .clip(job["geometry"])
    )
    water = job["prepared"].select("water_mask").selfMask().clip(job["geometry"])
    Map.addLayer(
        clear_rgb,
        {"min": 0.01, "max": 0.25, "gamma": 1.2},
        f"{map_id} Sentinel-2 RGB | {date_label}",
        False,
    )
    Map.addLayer(water, {"palette": ["00FFFF"]}, f"{map_id} supported water", False)
    Map.addLayer(
        ee.Image().paint(job["geometry"], 1, 2),
        {"palette": ["yellow"]},
        f"{map_id} reviewed polygon",
        False,
    )

Map.addLayerControl()
Map

## 8. Station-Point Feature Preview and Optional Training-Range Audit

In [ ]:
preview_frames = []
for map_id, job in jobs.items():
    sampled = job["stack"].sampleRegions(
        collection=job["points"],
        properties=["station_canonical", "coordinate_waterbody"],
        scale=EXPORT_SCALE_M,
        geometries=False,
        tileScale=4,
    ).getInfo()
    rows = [feature["properties"] for feature in sampled.get("features", [])]
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame.insert(0, "map_id", map_id)
        frame.insert(3, "sat_date", pd.Timestamp(job["row"]["sat_date"]).date().isoformat())
        preview_frames.append(frame)

station_feature_preview = pd.concat(preview_frames, ignore_index=True) if preview_frames else pd.DataFrame()
station_feature_preview.to_csv(
    OUTPUT_DIR / "station_feature_preview.csv", index=False, encoding="utf-8-sig"
)
print("Sampled station rows:", len(station_feature_preview))
display(station_feature_preview.head(30).round(5))

if MODEL_DATA_PATH is not None and not station_feature_preview.empty:
    training = pd.read_csv(MODEL_DATA_PATH, low_memory=False)
    common_numeric = [
        column for column in expected_schema
        if column in training.columns and column in station_feature_preview.columns
        and pd.api.types.is_numeric_dtype(pd.to_numeric(training[column], errors="coerce"))
    ]
    audits = []
    for column in common_numeric:
        train_values = pd.to_numeric(training[column], errors="coerce").dropna()
        map_values = pd.to_numeric(station_feature_preview[column], errors="coerce").dropna()
        if train_values.empty or map_values.empty:
            continue
        p01, p99 = train_values.quantile([0.01, 0.99])
        audits.append({
            "feature": column,
            "training_p01": p01,
            "training_p99": p99,
            "map_min": map_values.min(),
            "map_max": map_values.max(),
            "map_fraction_outside_training_p01_p99": ((map_values < p01) | (map_values > p99)).mean(),
        })
    feature_range_audit = pd.DataFrame(audits).sort_values(
        "map_fraction_outside_training_p01_p99", ascending=False
    )
    feature_range_audit.to_csv(OUTPUT_DIR / "map_feature_range_audit.csv", index=False, encoding="utf-8-sig")
    display(feature_range_audit.head(30).round(5))
else:
    print("Model dataset not found: schema and station previews were created, but training-range audit was skipped.")

## 9. Export All Feature Stacks

Exports are independent Earth Engine tasks. A failed area can be rerun without changing the manifest or other areas. Output filenames use stable ASCII `map_id` values; the Thai water-body name remains in the manifest.

In [ ]:
export_tasks = {}

if START_EXPORT_TASKS:
    for map_id, job in jobs.items():
        if EXPORT_MAP_IDS is not None and map_id not in set(EXPORT_MAP_IDS):
            continue
        row = job["row"]
        date_text = pd.Timestamp(row["sat_date"]).strftime("%Y%m%d")

        feature_prefix = f"{map_id}_{date_text}_moving_window_features"
        feature_task = ee.batch.Export.image.toDrive(
            image=job["stack"],
            description=feature_prefix,
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=feature_prefix,
            region=job["geometry"],
            scale=EXPORT_SCALE_M,
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={"cloudOptimized": True, "noData": -9999},
        )
        feature_task.start()
        export_tasks[(map_id, "features")] = feature_task

        rgb_prefix = f"{map_id}_{date_text}_sentinel2_rgb"
        clear_rgb = (
            job["prepared"].select(["B4", "B3", "B2"])
            .updateMask(job["prepared"].select("valid_mask"))
            .clip(job["geometry"])
            .toFloat()
        )
        rgb_task = ee.batch.Export.image.toDrive(
            image=clear_rgb,
            description=rgb_prefix,
            folder=RGB_DRIVE_EXPORT_FOLDER,
            fileNamePrefix=rgb_prefix,
            region=job["geometry"],
            scale=EXPORT_SCALE_M,
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={"cloudOptimized": True, "noData": -9999},
        )
        rgb_task.start()
        export_tasks[(map_id, "sentinel2_rgb")] = rgb_task
        print("Started:", map_id, "features=", feature_task.id, "rgb=", rgb_task.id)
else:
    print("START_EXPORT_TASKS=False: no Earth Engine export was started.")

In [ ]:
def task_status_table(tasks):
    rows = []
    for (map_id, task_type), task in tasks.items():
        status = task.status()
        rows.append({
            "map_id": map_id,
            "task_type": task_type,
            "task_id": task.id,
            "state": status.get("state"),
            "error_message": status.get("error_message"),
        })
    return pd.DataFrame(rows).sort_values(["map_id", "task_type"]).reset_index(drop=True)


if export_tasks:
    display(task_status_table(export_tasks))

if WAIT_FOR_EXPORTS and export_tasks:
    deadline = time.time() + MAX_WAIT_MINUTES * 60
    terminal = {"COMPLETED", "FAILED", "CANCELLED"}
    while time.time() < deadline:
        status_df = task_status_table(export_tasks)
        display(status_df)
        if status_df["state"].isin(terminal).all():
            break
        time.sleep(POLL_SECONDS)
    final_export_status = task_status_table(export_tasks)
    final_export_status.to_csv(OUTPUT_DIR / "export_task_status.csv", index=False, encoding="utf-8-sig")
    display(final_export_status)

In [ ]:
# Verify exported files and write the hand-off contract consumed by Notebook 04.
import rasterio

FEATURE_EXPORT_DIR = MYDRIVE / DRIVE_EXPORT_FOLDER
RGB_EXPORT_DIR = MYDRIVE / RGB_DRIVE_EXPORT_FOLDER
manifest_map_ids = set(manifest["map_id"].astype(str))
expected_map_ids = (
    manifest_map_ids
    if EXPORT_MAP_IDS is None
    else manifest_map_ids.intersection({str(value) for value in EXPORT_MAP_IDS})
)
if not expected_map_ids:
    raise ValueError("EXPORT_MAP_IDS did not match any mapping-manifest rows.")
selected_manifest = manifest.loc[manifest["map_id"].astype(str).isin(expected_map_ids)].copy()


def exported_raster_inventory(root, name_token, expected_band_count):
    rows = []
    paths = sorted(root.rglob("*.tif")) if root.exists() else []
    for path in paths:
        if name_token not in path.name:
            continue
        match = re.match(r"^(WB\d{2})_", path.name)
        if match is None:
            continue
        try:
            with rasterio.open(path) as src:
                rows.append({
                    "map_id": match.group(1), "file": str(path), "band_count": src.count,
                    "width": src.width, "height": src.height, "crs": str(src.crs),
                    "valid": src.count == expected_band_count and src.crs is not None,
                    "error": None,
                })
        except Exception as exc:
            rows.append({
                "map_id": match.group(1), "file": str(path), "band_count": None,
                "width": None, "height": None, "crs": None,
                "valid": False, "error": str(exc),
            })
    columns = ["map_id", "file", "band_count", "width", "height", "crs", "valid", "error"]
    return pd.DataFrame(rows, columns=columns)


feature_inventory = exported_raster_inventory(
    FEATURE_EXPORT_DIR, "moving_window_features", len(expected_schema)
)
rgb_inventory = exported_raster_inventory(RGB_EXPORT_DIR, "sentinel2_rgb", 3)
feature_inventory.to_csv(OUTPUT_DIR / "feature_geotiff_inventory.csv", index=False, encoding="utf-8-sig")
rgb_inventory.to_csv(OUTPUT_DIR / "sentinel2_rgb_inventory.csv", index=False, encoding="utf-8-sig")

feature_ids = set(feature_inventory.loc[feature_inventory["valid"], "map_id"])
rgb_ids = set(rgb_inventory.loc[rgb_inventory["valid"], "map_id"])
station_expected = len(
    station_inventory.loc[
        station_inventory["coordinate_waterbody"].isin(selected_manifest["coordinate_waterbody"])
    ]
)
station_found = len(
    station_feature_preview.loc[
        station_feature_preview["map_id"].astype(str).isin(expected_map_ids)
    ]
)
qa_map_ids = set(station_feature_preview.get("map_id", pd.Series(dtype=str)).astype(str))

if export_tasks:
    current_task_status = task_status_table(export_tasks)
    tasks_complete = current_task_status["state"].eq("COMPLETED").all()
    task_detail = current_task_status["state"].value_counts().to_dict()
else:
    tasks_complete = expected_map_ids.issubset(feature_ids) and expected_map_ids.issubset(rgb_ids)
    task_detail = "No in-memory tasks; completion verified from exported GeoTIFFs."

range_audit_path = OUTPUT_DIR / "map_feature_range_audit.csv"
checks = [
    {
        "check": "reviewed_polygon_complete",
        "passed": expected_map_ids.issubset(set(polygon_audit["map_id"])),
        "detail": f"{len(set(polygon_audit['map_id']) & expected_map_ids)}/{len(expected_map_ids)} polygon AOIs",
    },
    {
        "check": "same_62_band_schema",
        "passed": len(expected_schema) == 62 and len(set(expected_schema)) == 62,
        "detail": f"{len(expected_schema)} bands",
    },
    {
        "check": "all_station_rows_retained",
        "passed": station_found == station_expected,
        "detail": f"{station_found}/{station_expected} rows",
    },
    {
        "check": "every_map_has_pixel_QA_sample",
        "passed": expected_map_ids.issubset(qa_map_ids),
        "detail": f"{len(qa_map_ids & expected_map_ids)}/{len(expected_map_ids)} maps",
    },
    {
        "check": "training_range_audit_available",
        "passed": range_audit_path.exists() and range_audit_path.stat().st_size > 0,
        "detail": str(range_audit_path) if range_audit_path.exists() else "Set MODEL_DATA_PATH_OVERRIDE",
    },
    {
        "check": "earth_engine_exports_completed",
        "passed": bool(tasks_complete),
        "detail": str(task_detail),
    },
    {
        "check": "feature_geotiffs_valid",
        "passed": expected_map_ids.issubset(feature_ids),
        "detail": f"{len(feature_ids & expected_map_ids)}/{len(expected_map_ids)} logical maps",
    },
    {
        "check": "sentinel2_rgb_geotiffs_valid",
        "passed": expected_map_ids.issubset(rgb_ids),
        "detail": f"{len(rgb_ids & expected_map_ids)}/{len(expected_map_ids)} logical maps",
    },
]

notebook03_readiness = pd.DataFrame(checks)
notebook03_readiness.to_csv(
    OUTPUT_DIR / "notebook03_readiness.csv", index=False, encoding="utf-8-sig"
)
display(notebook03_readiness)
print("NOTEBOOK_03_READY =", bool(notebook03_readiness["passed"].all()))
print("Copy both GeoTIFF export folders and the OUTPUT_DIR audit files to the local moving_window folder.")

## Completion Gate Before Model Prediction

Proceed only when:

- every manifest `map_id` has exactly one reviewed Polygon/MultiPolygon;
- every job uses `aoi_source=reviewed_drawn_polygon`;
- every feature stack retains the locked 62-band schema;
- station-point previews contain valid moving-window features;
- both the feature-stack and exact-date Sentinel-2 RGB export tasks are `COMPLETED` for every map;
- the optional range audit is reviewed for severe extrapolation.

Notebook 04 must use the new polygon feature stacks and matching Sentinel-2 RGB files. It must not reuse the earlier circular-AOI predictions.